# FPS v1 Shadow Analysis — Setup Class Evaluation

**Purpose:** Evaluate setup-class performance from shadow logs.  
**Data source:** `logs/execution/futures_permit_surface_shadow.jsonl`  
**Authority:** None — this notebook is read-only research.  

## Metrics tracked
1. Permit rate per symbol / regime (sparsity invariant)
2. Expectancy per setup class (forward-looking PnL attribution)
3. Fee-clearing rate (gate 3 pass/fail)
4. False-positive density (permits that would have lost)
5. Setup class overlap / mutual exclusivity
6. Regime-conditional permit distribution

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

import pandas as pd
import numpy as np

# Ensure project root is on path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from execution.futures_permit_surface_v1 import (
    PERMIT_CANDIDATE, ABSTAIN, DENY_STRUCTURAL,
    _VALID_SETUP_CLASSES, compute_sparsity, SparsityReport,
)

SHADOW_LOG = PROJECT_ROOT / "logs" / "execution" / "futures_permit_surface_shadow.jsonl"
print(f"Shadow log: {SHADOW_LOG}")
print(f"Exists: {SHADOW_LOG.exists()}")

## 1. Load Shadow Log

In [ ]:
def load_shadow_log(path: Path = SHADOW_LOG) -> pd.DataFrame:
    """Load shadow JSONL into DataFrame. Returns empty df if no data."""
    if not path.exists():
        print(f"No shadow log found at {path}")
        print("Run the executor with FPS v1 enabled to generate data.")
        return pd.DataFrame()
    
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
    
    if not records:
        print("Shadow log exists but is empty.")
        return pd.DataFrame()
    
    df = pd.DataFrame(records)
    if "ts" in df.columns:
        df["datetime"] = pd.to_datetime(df["ts"], unit="s", utc=True)
    print(f"Loaded {len(df)} shadow records")
    print(f"Date range: {df['datetime'].min()} → {df['datetime'].max()}" if 'datetime' in df.columns else "")
    return df

df = load_shadow_log()
if not df.empty:
    df.head()

## 2. Sparsity Invariant Check

**Healthy range:** 2–10% permit rate  
**Alert thresholds:** >15% (drift/too loose), <1% (dead surface)

In [ ]:
report = compute_sparsity(SHADOW_LOG)

print("=" * 60)
print("SPARSITY INVARIANT REPORT")
print("=" * 60)
print(f"Total evaluations:  {report.total_evals:,}")
print(f"Permit count:       {report.permit_count:,}")
print(f"Global permit rate: {report.permit_rate:.2%}")
print()

if report.alerts:
    print("⚠️  ALERTS:")
    for a in report.alerts:
        print(f"  • {a}")
else:
    print("✅ No alerts — sparsity within healthy bounds.")

print()
print("--- Per-symbol permit rates ---")
for sym, rate in sorted(report.by_symbol.items(), key=lambda x: -x[1]):
    flag = " ⚠️" if rate > 0.15 else (" 💀" if 0 < rate < 0.01 else "")
    print(f"  {sym:12s}  {rate:6.2%}{flag}")

print()
print("--- Per-regime permit rates ---")
for rg, rate in sorted(report.by_regime.items(), key=lambda x: -x[1]):
    flag = " ⚠️" if rate > 0.15 else ""
    print(f"  {rg:20s}  {rate:6.2%}{flag}")

print()
print("--- Setup class distribution ---")
for sc, count in sorted(report.by_setup_class.items(), key=lambda x: -x[1]):
    pct = count / report.permit_count if report.permit_count else 0
    print(f"  {sc:35s}  {count:5d}  ({pct:5.1%})")

if report.overlap_violations:
    print()
    print(f"--- Overlap violations: {len(report.overlap_violations)} ---")
    for v in report.overlap_violations[:10]:
        print(f"  {v['symbol']} @ {v['ts']:.1f}: {v['classes']}")

## 3. Verdict Distribution

In [ ]:
if not df.empty:
    verdict_counts = df["verdict"].value_counts()
    verdict_pcts = df["verdict"].value_counts(normalize=True)
    
    print("Verdict Distribution:")
    print("-" * 40)
    for v in [PERMIT_CANDIDATE, ABSTAIN, DENY_STRUCTURAL]:
        count = verdict_counts.get(v, 0)
        pct = verdict_pcts.get(v, 0.0)
        print(f"  {v:20s}  {count:6d}  ({pct:6.2%})")
    
    # Verdict by symbol
    print()
    print("Verdict × Symbol (permit rate):")
    print("-" * 40)
    pivot = pd.crosstab(df["symbol"], df["verdict"], normalize="index")
    if PERMIT_CANDIDATE in pivot.columns:
        print(pivot.sort_values(PERMIT_CANDIDATE, ascending=False).to_string())
    else:
        print("No permits yet.")
else:
    print("No data to analyse.")

## 4. Setup Class Performance

Per-class metrics:
- Fire count & rate
- Regime distribution when firing
- Direction bias
- Fee-clearing rate

In [ ]:
if not df.empty and PERMIT_CANDIDATE in df["verdict"].values:
    permits = df[df["verdict"] == PERMIT_CANDIDATE].copy()
    
    print(f"Total permits: {len(permits)}")
    print()
    
    for sc in sorted(permits["setup_class"].unique()):
        subset = permits[permits["setup_class"] == sc]
        print(f"\n{'=' * 60}")
        print(f"Setup class: {sc}")
        print(f"{'=' * 60}")
        print(f"  Count:      {len(subset)}")
        print(f"  Share:      {len(subset)/len(permits):.1%}")
        
        # Direction bias
        dir_counts = subset["direction"].value_counts()
        print(f"  Direction:  {dict(dir_counts)}")
        
        # Regime distribution
        regime_counts = subset["regime_current"].value_counts()
        print(f"  Regimes:    {dict(regime_counts)}")
        
        # ATR stats
        if "atr_pct" in subset.columns:
            atr = subset["atr_pct"]
            print(f"  ATR pct:    mean={atr.mean():.4f}  med={atr.median():.4f}  std={atr.std():.4f}")
        
        # Volume Z stats
        if "volume_z" in subset.columns:
            vz = subset["volume_z"]
            print(f"  Volume Z:   mean={vz.mean():.2f}  med={vz.median():.2f}  std={vz.std():.2f}")
        
        # Regime confidence
        if "regime_confidence" in subset.columns:
            rc = subset["regime_confidence"]
            print(f"  Confidence: mean={rc.mean():.3f}  med={rc.median():.3f}  std={rc.std():.3f}")
else:
    print("No permits to analyse — setup classes have not fired yet.")

## 5. Fee-Clearing Analysis (Gate 3)

In [ ]:
if not df.empty:
    # Denials specifically from fee bridge
    fee_denials = df[df["deny_reason"] == "fee_bridge_insufficient"]
    
    # Records that passed gate 1 (have a setup_class)
    gate1_pass = df[df["setup_class"].notna()]
    
    gate1_count = len(gate1_pass)
    fee_deny_count = len(fee_denials)
    permit_count = len(df[df["verdict"] == PERMIT_CANDIDATE])
    
    print("Fee Bridge Analysis (Gate 3):")
    print("-" * 40)
    print(f"  Evaluations reaching gate 3:  {gate1_count}")
    print(f"  Fee denials:                  {fee_deny_count}")
    print(f"  Fee-clearing rate:            {(gate1_count - fee_deny_count) / gate1_count:.1%}" if gate1_count else "  N/A")
    
    if not fee_denials.empty:
        print()
        print("Fee denials by symbol:")
        print(fee_denials["symbol"].value_counts().to_string())
        print()
        print("Fee denials by setup class:")
        print(fee_denials["setup_class"].value_counts().to_string())
        print()
        print(f"ATR of fee-denied (mean): {fee_denials['atr_pct'].mean():.5f}")
        print(f"ATR of permits (mean):    {df[df['verdict']==PERMIT_CANDIDATE]['atr_pct'].mean():.5f}" if permit_count else "")
else:
    print("No data.")

## 6. Temporal Analysis — Permit Rate Over Time

In [ ]:
if not df.empty and "datetime" in df.columns:
    # Hourly permit rate
    df["hour"] = df["datetime"].dt.floor("h")
    hourly = df.groupby("hour").agg(
        total=("verdict", "count"),
        permits=("verdict", lambda x: (x == PERMIT_CANDIDATE).sum()),
    )
    hourly["permit_rate"] = hourly["permits"] / hourly["total"]
    
    print("Hourly permit rate (last 24h):")
    print("-" * 50)
    tail = hourly.tail(24)
    for idx, row in tail.iterrows():
        bar = "█" * int(row["permit_rate"] * 100)
        flag = " ⚠️" if row["permit_rate"] > 0.15 else ""
        print(f"  {idx}  {row['permit_rate']:6.2%}  [{row['permits']:3.0f}/{row['total']:3.0f}]  {bar}{flag}")
    
    print()
    print(f"Overall hourly permit rate: mean={hourly['permit_rate'].mean():.2%}  std={hourly['permit_rate'].std():.2%}")
    
    # Check for drift: is permit rate trending up?
    if len(hourly) >= 10:
        first_half = hourly.iloc[:len(hourly)//2]["permit_rate"].mean()
        second_half = hourly.iloc[len(hourly)//2:]["permit_rate"].mean()
        drift = second_half - first_half
        print(f"Drift check: first_half={first_half:.2%}  second_half={second_half:.2%}  Δ={drift:+.2%}")
        if abs(drift) > 0.05:
            print("  ⚠️  Significant permit rate drift detected!")
        else:
            print("  ✅ Permit rate stable.")
else:
    print("No temporal data available.")

## 6b. Temporal Clustering (Burst Risk)

**Independence check:** Even if global permit rate is healthy, clustered bursts break statistical validity.

Alert thresholds:
- **>30% permit rate** within any 1h or 4h window → burst clustering
- **>5 consecutive permits** → non-independent signal

In [ ]:
from execution.futures_permit_surface_v1 import compute_burst_risk, BurstReport

burst = compute_burst_risk(SHADOW_LOG)

print("=" * 60)
print("TEMPORAL CLUSTERING REPORT")
print("=" * 60)
print(f"Total permits:         {burst.total_permits}")
print(f"Windows checked (1h):  {burst.windows_checked_1h}")
print(f"Windows checked (4h):  {burst.windows_checked_4h}")
print(f"Peak 1h permit rate:   {burst.max_rate_1h:.2%}")
print(f"Peak 4h permit rate:   {burst.max_rate_4h:.2%}")
print(f"Longest consecutive:   {burst.max_consecutive}")
print()

if burst.alerts:
    print("⚠️  ALERTS:")
    for a in burst.alerts:
        print(f"  • {a}")
else:
    print("✅ No temporal clustering detected.")

if burst.burst_windows_1h:
    print()
    print("--- 1h burst windows ---")
    for w in burst.burst_windows_1h[:10]:
        print(f"  [{w['window_start']:.0f} → {w['window_end']:.0f}]  "
              f"{w['permits']}/{w['total']} = {w['rate']:.1%}")

if burst.burst_windows_4h:
    print()
    print("--- 4h burst windows ---")
    for w in burst.burst_windows_4h[:10]:
        print(f"  [{w['window_start']:.0f} → {w['window_end']:.0f}]  "
              f"{w['permits']}/{w['total']} = {w['rate']:.1%}")

if burst.consecutive_streaks:
    print()
    print("--- Consecutive permit streaks ---")
    for s in burst.consecutive_streaks[:10]:
        print(f"  {s['symbol']}  length={s['length']}  "
              f"[{s['start_ts']:.0f} → {s['end_ts']:.0f}]")

## 7. Regime × Setup Class Heatmap (Text)

In [ ]:
if not df.empty and PERMIT_CANDIDATE in df["verdict"].values:
    permits = df[df["verdict"] == PERMIT_CANDIDATE]
    
    ct = pd.crosstab(permits["regime_current"], permits["setup_class"])
    print("Setup class × Regime (permit counts):")
    print(ct.to_string())
    print()
    
    # Check: each setup class should fire predominantly in its expected regime
    expected_regimes = {
        "REGIME_TRANSITION_BREAK": {"TREND_UP", "TREND_DOWN", "BREAKOUT"},
        "POST_CRISIS_RESTABILIZATION": {"TREND_UP", "TREND_DOWN", "MEAN_REVERT", "BREAKOUT"},
        "DISLOCATION_REVERSION": {"MEAN_REVERT"},
        "BREAKOUT_RETEST_CONFIRM": {"BREAKOUT"},
        "LIQUIDITY_VACUUM_RECLAIM": {"TREND_UP", "TREND_DOWN", "MEAN_REVERT", "BREAKOUT"},
    }
    
    print("Regime alignment check:")
    for sc in ct.columns:
        expected = expected_regimes.get(sc, set())
        actual = set(ct.index[ct[sc] > 0])
        unexpected = actual - expected
        if unexpected:
            print(f"  ⚠️  {sc}: fires in unexpected regimes {unexpected}")
        else:
            print(f"  ✅ {sc}: fires only in expected regimes {actual}")
else:
    print("No permits — cannot build heatmap.")

## 8. Forward Attribution Stub

Once position data is available, this section will join FPS permits with  
realized PnL to compute per-setup-class expectancy.

**Required fields:** `snapshot_hash` (join key), realized PnL from episode ledger.

In [ ]:
# --- Forward PnL Attribution (stub) ---
# This requires joining FPS shadow events with realized episode PnL.
# The join key is (symbol, timestamp window) or snapshot_hash.
#
# Pseudocode:
#   fps_permits = df[df.verdict == PERMIT_CANDIDATE]
#   episodes = load_episode_ledger()  # from logs/state/positions_ledger.json
#   joined = fps_permits.merge(episodes, on=["symbol"], how="left")
#   # Filter to episodes that started within ±15min of permit
#   joined = joined[abs(joined.entry_ts - joined.ts) < 900]
#   expectancy = joined.groupby("setup_class")["realized_pnl"].agg(["mean", "count", "std"])
#
# Key questions to answer:
# 1. Which setup classes have positive expected value?
# 2. Is the edge persistent across regimes?
# 3. Does fee-clearing actually predict profitability?

print("Forward attribution not yet available.")
print("Requires: shadow log data + episode ledger with realized PnL.")
print()
print("Evaluation criteria for each setup class:")
print("  • Positive mean PnL (after fees)")
print("  • Win rate > 45%")
print("  • Sharpe > 0.5 on per-trade basis")
print("  • Persistent across at least 2 regime types")
print("  • Minimum 30 observations before declaring significance")

## 9. Summary & Next Actions

In [ ]:
print("=" * 60)
print("FPS v1 SHADOW ANALYSIS SUMMARY")
print("=" * 60)

if report.total_evals == 0:
    print("\n📊 Status: NO DATA")
    print("\nThe surface has not been evaluated yet.")
    print("Enable FPS v1 in the executor loop to begin shadow collection.")
    print("\nExpected timeline:")
    print("  Week 1: Collect baseline permit rates")
    print("  Week 2: First sparsity check + regime coverage")
    print("  Week 3+: Forward PnL attribution (requires episode joins)")
else:
    print(f"\n📊 Status: {report.total_evals:,} evaluations collected")
    print(f"   Permit rate: {report.permit_rate:.2%}")
    print(f"   Active setup classes: {len(report.by_setup_class)}")
    print(f"   Alerts: {len(report.alerts)}")
    
    print("\n--- Decision framework ---")
    for sc, count in sorted(report.by_setup_class.items(), key=lambda x: -x[1]):
        print(f"   {sc}: {count} permits")
        if count < 10:
            print(f"     → Insufficient data (need ≥30 for significance)")
        elif count < 30:
            print(f"     → Approaching significance threshold")
        else:
            print(f"     → Ready for forward attribution")
    
    print("\n--- Action items ---")
    if any("DRIFT" in a for a in report.alerts):
        print("  🔴 DRIFT detected — tighten setup class conditions")
    if any("DEAD" in a for a in report.alerts):
        print("  🟡 Dead surface — loosen conditions or extend data window")
    if any("OVERLAP" in a for a in report.alerts):
        print("  🟡 Overlap detected — review mutual exclusivity of setup classes")
    if not report.alerts:
        print("  ✅ All invariants healthy — continue collection")